# Training Notebook

Train BiLSTM or Transformer models with optional cross-validation.

In [ ]:
import sys
sys.path.insert(0, '..')

from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, RichProgressBar
from src.data import GestureDataModule
from src.models import BiLSTMModule, TransformerModule
from src.training import CrossValidator
from src.training.callbacks import BestModelCallback

## Configuration

In [ ]:
# Choose model: 'bilstm' or 'transformer'
MODEL = 'bilstm'

# Training settings
EPOCHS = 50
PATIENCE = 15
BATCH_SIZE = 32
SEED = 42

# Cross-validation
USE_CV = True
N_FOLDS = 5

# Data (None = auto-download from HuggingFace)
DATA_PATH = '../data/DYLEM-GRID'

seed_everything(SEED)

## Model Hyperparameters

In [ ]:
# BiLSTM hyperparameters
BILSTM_PARAMS = {
    'hidden_size': 64,
    'num_layers': 2,
    'dropout': 0.15,
    'learning_rate': 0.002,
    'optimizer': 'nadam',
    'use_attention': True
}

# Transformer hyperparameters
TRANSFORMER_PARAMS = {
    'd_model': 64,
    'nhead': 8,
    'num_layers': 2,
    'dim_feedforward': 128,
    'dropout': 0.1,
    'learning_rate': 0.001,
    'optimizer': 'nadam',
    'pooling': 'mean'
}

PARAMS = BILSTM_PARAMS if MODEL == 'bilstm' else TRANSFORMER_PARAMS
MODEL_CLASS = BiLSTMModule if MODEL == 'bilstm' else TransformerModule

## Load Data

In [ ]:
dm = GestureDataModule(
    data_path=DATA_PATH,
    batch_size=BATCH_SIZE,
    seed=SEED
)
dm.setup()

print(f'Input size: {dm.input_size}')
print(f'Classes: {dm.class_names}')
print(f'Train samples: {len(dm.train_dataset)}')
print(f'Val samples: {len(dm.val_dataset)}')

## Training

In [ ]:
if USE_CV:
    print(f'\n{N_FOLDS}-Fold Cross-Validation')
    print('=' * 50)
    
    cv = CrossValidator(
        MODEL_CLASS, dm, N_FOLDS,
        trainer_kwargs={'max_epochs': EPOCHS, 'accelerator': 'auto'}
    )
    results = cv.run(PARAMS, patience=PATIENCE)
    
else:
    print(f'\nTraining {MODEL.upper()}')
    print('=' * 50)
    
    model = MODEL_CLASS(
        input_size=dm.input_size,
        num_classes=dm.num_classes,
        **PARAMS
    )
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
    
    callbacks = [
        BestModelCallback(),
        EarlyStopping('val_acc', mode='max', patience=PATIENCE),
        ModelCheckpoint('../models/checkpoints', f'{MODEL}_best', monitor='val_acc', mode='max'),
        RichProgressBar()
    ]
    
    trainer = Trainer(max_epochs=EPOCHS, callbacks=callbacks, accelerator='auto')
    trainer.fit(model, dm)

## Results

In [ ]:
if USE_CV:
    import matplotlib.pyplot as plt
    
    accs = [r.val_acc for r in results.fold_results]
    plt.figure(figsize=(8, 4))
    plt.bar(range(1, len(accs)+1), accs)
    plt.axhline(results.mean_accuracy, color='red', linestyle='--', label=f'Mean: {results.mean_accuracy:.4f}')
    plt.xlabel('Fold')
    plt.ylabel('Accuracy')
    plt.title(f'{MODEL.upper()} Cross-Validation Results')
    plt.legend()
    plt.ylim(0, 1.05)
    plt.show()
else:
    print(f'\nBest validation accuracy: {trainer.callback_metrics.get("val_acc", 0):.4f}')
    print(f'Model saved to: ../models/checkpoints/{MODEL}_best.ckpt')